In [36]:
from statsmodels.tsa.statespace.sarimax import SARIMAXResults
import pandas as pd
import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta

model_path = "models/sarimax_AECO.pkl"   # or "./models/sarimax_AECO.pkl"
model_AECO = SARIMAXResults.load(model_path)

weather_path = "temperature/live_weather_AECO.csv"
weather_AECO = pd.read_csv(weather_path, parse_dates=["time"])   # time column exists
weather_AECO = weather_AECO.set_index("time")

cols = weather_AECO.iloc[:, [3, 2, 0, 4]]
temperature = cols.values.flatten()

temp_vec = temperature  # your 96-long vector
df_temp = pd.DataFrame({"temp": temp_vec})

Tbase = 18.0
df_temp["CDH"] = (df_temp["temp"] - Tbase).clip(lower=0)
df_temp["HDH"] = (Tbase - df_temp["temp"]).clip(lower=0)

tz = pytz.timezone("America/New_York")
now = datetime.now(tz)
today_00 = now.replace(hour=0, minute=0, second=0, microsecond=0)
start_time = today_00 - timedelta(days=2)
start_time = start_time.replace(tzinfo=None)
df_temp["datetime"] = pd.date_range(start=start_time, periods=96, freq="H")
df_temp["dow"] = df_temp["datetime"].dt.dayofweek
df_temp = pd.get_dummies(df_temp, columns=["dow"], prefix="dow", dtype=float, drop_first=True)

exog_cols = ["CDH","HDH","dow_1","dow_2","dow_3","dow_4","dow_5","dow_6"]

for col in exog_cols:
    if col not in df_temp.columns:
        df_temp[col] = 0.0

exog_future = df_temp[exog_cols]

forecast = model_AECO.get_forecast(steps=96, exog=exog_future)
mean_forecast = forecast.predicted_mean

last_24 = mean_forecast[-24:].values
last_24
peak_hour = last_24.argmax()   # integer between 0 and 23
peak_hour

/var/folders/jn/yj91f36d1q5czmlvzf8lf2f40000gr/T/ipykernel_43494/696014732.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_temp["datetime"] = pd.date_range(start=start_time, periods=96, freq="H")
/Users/seyonghw/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/seyonghw/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


np.int64(18)